# Phase 1 and Phase 2 CIFAR-10 experiments

This notebook records the baseline and TENT experiments in reproducible, batch-wise CSV files. Run it from the project root (or open it in VS Code with this project as the workspace). It uses the trained ResNet18 checkpoint and CIFAR-10/CIFAR-10-C data already stored under `data/`.

| ID | Experiment | What it measures |
|---|---|---|
| E1 | Clean CIFAR-10 classification | Source model on the clean test set |
| E2 | CIFAR-10-C corruption evaluation | Source model on a selected corruption |
| E3 | Entropy calculation | Entropy for uniform and confident predictions |
| E4 | TENT on Gaussian noise | Online adaptation on Gaussian noise |
| E5 | TENT on blur | Online adaptation on Gaussian blur |
| E6 | TENT on fog | Online adaptation on fog |
| E7 | Adaptation learning rates | Repeat the same corruption from a fresh checkpoint for each rate |

E4–E7 are separate cells so you can begin with Gaussian noise before running other corruptions. Each run scores a batch before its TENT update. Labels are only used to compute metrics, never by the adaptation function. CSV rows include batch number, adaptation entropy loss, accuracy, predicted counts per class, marginal entropy, and learning rate.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch
import torch.nn.functional as F

# Find the project root whether the notebook is opened from the root or experiments/.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / "models" / "resnet.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the project workspace.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from datasets.cifar_dataset import create_dataloaders
from datasets.corruptions import (
    CIFAR10C_ROOT,
    CLASS_NAMES,
    available_corruptions,
    create_cifar10c_loader,
)
from models.resnet import create_model
from phase2_tent.adapt import adapt_batch, create_tent_optimizer
from phase2_tent.entropy import entropy_loss

CHECKPOINT = PROJECT_ROOT / "models" / "checkpoints" / "resnet18_cifar10.pth"
BATCH_SIZE = 32
NUM_WORKERS = 0  # Reliable for Windows notebooks; increase if your system benefits.
DEFAULT_CORRUPTION = "gaussian_noise"
DEFAULT_SEVERITY = 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASELINE_RESULTS = PROJECT_ROOT / "experiments" / "baseline_results"
TENT_RESULTS = PROJECT_ROOT / "experiments" / "tent_results"

if not CHECKPOINT.is_file():
    raise FileNotFoundError(f"Train the baseline first; checkpoint not found: {CHECKPOINT}")
print(f"Project: {PROJECT_ROOT}\nDevice: {DEVICE}\nCheckpoint: {CHECKPOINT}")
print("Available CIFAR-10-C corruptions:", available_corruptions(CIFAR10C_ROOT))

In [ ]:
def load_fresh_model():
    """Load a new model from the original checkpoint for an independent run."""
    model = create_model(num_classes=len(CLASS_NAMES)).to(DEVICE)
    state = torch.load(CHECKPOINT, map_location=DEVICE)
    model.load_state_dict(state)
    return model


def get_corruption_loader(corruption=DEFAULT_CORRUPTION, severity=DEFAULT_SEVERITY):
    return create_cifar10c_loader(
        corruption=corruption,
        severity=severity,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
    )


def evaluate_batches(
    model,
    loader,
    experiment_id,
    dataset_name,
    adapt=False,
    learning_rate=None,
    output_dir=BASELINE_RESULTS,
):
    """Evaluate every batch and save accuracy, entropy, and prediction counts to CSV."""
    optimizer = create_tent_optimizer(model, learning_rate=learning_rate) if adapt else None
    rows = []
    total_correct = 0
    total_samples = 0
    total_loss = 0.0
    total_adaptation_loss = 0.0

    for batch_number, (images, labels) in enumerate(loader, start=1):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        if adapt:
            # logits describe this batch before adapt_batch updates BN parameters.
            adaptation_loss, logits = adapt_batch(model, images, optimizer)
            total_adaptation_loss += adaptation_loss * labels.size(0)
        else:
            model.eval()
            with torch.inference_mode():
                logits = model(images)
            adaptation_loss = ""

        probabilities = F.softmax(logits, dim=1)
        marginal = probabilities.mean(dim=0)
        marginal_entropy = -(marginal * marginal.clamp_min(1e-8).log()).sum().item()
        predictions = logits.argmax(dim=1)
        correct = (predictions == labels).sum().item()
        batch_count = labels.size(0)
        total_correct += correct
        total_samples += batch_count
        total_loss += F.cross_entropy(logits, labels, reduction="sum").item()

        row = {
            "experiment_id": experiment_id,
            "dataset": dataset_name,
            "batch_number": batch_number,
            "batch_size": batch_count,
            "adaptation_loss": adaptation_loss,
            "accuracy_percent": 100.0 * correct / batch_count,
            "marginal_entropy": marginal_entropy,
            "learning_rate": learning_rate if adapt else "",
        }
        prediction_counts = torch.bincount(
            predictions.detach().cpu(), minlength=len(CLASS_NAMES)
        ).tolist()
        row.update(
            {f"predicted_{name}": count for name, count in zip(CLASS_NAMES, prediction_counts)}
        )
        rows.append(row)

    if total_samples == 0:
        raise ValueError("Cannot evaluate an empty data loader.")

    output_dir.mkdir(parents=True, exist_ok=True)
    safe_dataset_name = dataset_name.replace(" ", "_").replace(",", "")
    lr_suffix = f"_lr{learning_rate:g}" if adapt else ""
    csv_path = output_dir / f"{experiment_id}_{safe_dataset_name}{lr_suffix}.csv"
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    summary = {
        "experiment_id": experiment_id,
        "dataset": dataset_name,
        "accuracy_percent": 100.0 * total_correct / total_samples,
        "cross_entropy": total_loss / total_samples,
        "mean_adaptation_loss": total_adaptation_loss / total_samples if adapt else None,
        "learning_rate": learning_rate if adapt else None,
        "batches": len(rows),
        "samples": total_samples,
        "csv_path": str(csv_path),
    }
    return summary, pd.DataFrame(rows)


def print_summary(summary):
    print(pd.Series(summary).to_string())

## E1 — Clean CIFAR-10 classification

The source model is evaluated on the clean test set. Batch adaptation loss and learning rate are blank because this is ordinary source-model evaluation.

In [ ]:
_, clean_test_loader, _ = create_dataloaders(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
e1_summary, e1_batches = evaluate_batches(
    load_fresh_model(), clean_test_loader, "E1", "clean_cifar10",
    output_dir=BASELINE_RESULTS,
)
print_summary(e1_summary)
e1_batches.head()

## E2 — CIFAR-10-C source-model corruption evaluation

Change the corruption and severity to compare other conditions. This step does not adapt the model.

In [ ]:
corruption_loader = get_corruption_loader(DEFAULT_CORRUPTION, DEFAULT_SEVERITY)
e2_dataset_name = f"{DEFAULT_CORRUPTION}_severity_{DEFAULT_SEVERITY}"
e2_summary, e2_batches = evaluate_batches(
    load_fresh_model(), corruption_loader, "E2", e2_dataset_name,
    output_dir=BASELINE_RESULTS,
)
print_summary(e2_summary)
e2_batches.head()

## E3 — Entropy calculation

Uniform probabilities have higher entropy than confident probabilities. Entropy is measured in nats when using the natural logarithm.

In [ ]:
uniform_probabilities = torch.tensor([[0.5, 0.5]])
confident_probabilities = torch.tensor([[0.99, 0.01]])
uniform_entropy = entropy_loss(uniform_probabilities.log()).item()
confident_entropy = entropy_loss(confident_probabilities.log()).item()
print(f"Uniform [0.50, 0.50] entropy: {uniform_entropy:.4f}")
print(f"Confident [0.99, 0.01] entropy: {confident_entropy:.4f}")
assert uniform_entropy > confident_entropy

## E4 — TENT adaptation on Gaussian noise

Each batch is scored before its images are used for an unlabeled entropy-minimization update. The original checkpoint is loaded fresh for this run.

In [ ]:
DEFAULT_LR = 0.00025
e4_loader = get_corruption_loader("gaussian_noise", 1)
e4_summary, e4_batches = evaluate_batches(
    load_fresh_model(), e4_loader, "E4", "gaussian_noise_severity_1",
    adapt=True, learning_rate=DEFAULT_LR, output_dir=TENT_RESULTS,
)
print_summary(e4_summary)
e4_batches.head()

## E5 — TENT adaptation on blur

This uses CIFAR-10-C `gaussian_blur.npy` at severity 1. It starts again from the original checkpoint, not from E4's adapted weights.

In [ ]:
e5_loader = get_corruption_loader("gaussian_blur", 1)
e5_summary, e5_batches = evaluate_batches(
    load_fresh_model(), e5_loader, "E5", "gaussian_blur_severity_1",
    adapt=True, learning_rate=DEFAULT_LR, output_dir=TENT_RESULTS,
)
print_summary(e5_summary)
e5_batches.head()

## E6 — TENT adaptation on fog

This also starts from the original checkpoint.

In [ ]:
e6_loader = get_corruption_loader("fog", 1)
e6_summary, e6_batches = evaluate_batches(
    load_fresh_model(), e6_loader, "E6", "fog_severity_1",
    adapt=True, learning_rate=DEFAULT_LR, output_dir=TENT_RESULTS,
)
print_summary(e6_summary)
e6_batches.head()

## E7 — Compare adaptation learning rates

Each rate gets a fresh model and optimizer loaded from the original checkpoint. This ensures an experiment never inherits updates from a prior learning-rate run. Adjust the list if you want to test other values.

In [ ]:
LEARNING_RATES = [0.00005, 0.00025, 0.001]
e7_summaries = []
for rate in LEARNING_RATES:
    loader = get_corruption_loader(DEFAULT_CORRUPTION, DEFAULT_SEVERITY)
    summary, batch_metrics = evaluate_batches(
        load_fresh_model(), loader, "E7",
        f"{DEFAULT_CORRUPTION}_severity_{DEFAULT_SEVERITY}",
        adapt=True, learning_rate=rate, output_dir=TENT_RESULTS,
    )
    e7_summaries.append(summary)
pd.DataFrame(e7_summaries)

## Reading the batch CSV

- `adaptation_loss`: mean prediction entropy for the batch, blank for source-only evaluation.
- `accuracy_percent`: batch accuracy from predictions made before that batch's TENT update.
- `predicted_<class>`: number of batch predictions assigned to that class; the counts sum to `batch_size`.
- `marginal_entropy`: entropy of the average predicted class distribution for the batch. Lower indicates a more concentrated class distribution, not necessarily better accuracy.
- `learning_rate`: optimizer rate for TENT, blank for source-only runs.

CSV files are written under `experiments/baseline_results/` for E1/E2 and `experiments/tent_results/` for E4–E7. E3 is a small sanity check rather than a dataset evaluation, so it prints its two entropy values.